# Lab: Build a RAG System Over Your Dissertation

**Author:** Jason A. Minton, Ph.D.
**Purpose:** A hands-on RAG implementation.

---

## What you will build by the end

A working Retrieval-Augmented Generation system that lets you ask natural-language questions about J.A. Minton's dissertation and receive answers grounded in the text — with the ability to inspect which passages supported each answer. You will run this end-to-end in a single notebook.**Source document:** Minton, J. A. (2025). *Data Quality and Enterprise Architecture: An Analysis of the Awareness and Knowledge of Data Quality Artifacts, Principles, Practices, and Application in Enterprise Architecture* (Doctoral dissertation). University of Arkansas at Little Rock. ProQuest No. 31935662. PDF not included in this repository — available from the author on request (author holds full copyright).

## Why this matters

Two reasons that go beyond "RAG is on the AI Learning Guide list of things to know."

**First, this lab puts the dissertation's broader argument into practice.** The dissertation finds that Enterprise Architects and Technology Architects often lack awareness of whether their organizations are measuring data quality, and that gaps persist in how data quality frameworks and specifications are integrated into Enterprise Architecture practice — gaps with direct downstream consequences for any AI system the enterprise deploys. RAG is one of the principal technical solutions that helps address downstream data quality consequences by grounding LLM outputs in specific, authoritative source documents rather than letting the model reason from training data alone. Building a RAG system over the dissertation is a quiet form of operating on the primary recommendation the dissertation makes: that high-quality data and grounded data sources are the foundation for trustworthy data (including AI).

**Second, it converts a "working knowledge" claim into a defensible one.** After completing this lab and saving the notebook, you can speak to RAG from direct implementation rather than from study alone. Questions about RAG will sound like *"I built a RAG system over a dissertation as a learning exercise — happy to walk through how I handled chunking, embedding choice, and citation tracking"* rather than *"I've read about RAG."*

## How to use this lab

Work the cells in order. Each part has three kinds of cells: prose explanations, runnable code, and reflection prompts. **Do not skip the reflection prompts.** They are where the conceptual understanding happens; the code only works if the concepts hold. Total focused time is roughly 90 minutes, comfortably split across two or three sessions.

The lab is designed to run in **Google Colab** (free, no local setup needed), though it will also run in a local Jupyter installation. Where Colab-specific instructions appear, they are clearly marked.

---

## Setup, before you begin

You need three things in place before running the first code cell.

**1. The dissertation PDF, accessible to the notebook.**
In Colab, open the file browser panel on the left (folder icon), and drag your dissertation PDF into the file area. By default, files upload to `/content/`. Whatever you name the file, note the exact filename — you will reference it in the next cell. The default in this notebook is `dissertation.pdf`; rename your upload to that, or change the variable in the code cell below.

**2. An Anthropic API key.**
Sign up at [console.anthropic.com](https://console.anthropic.com/) and generate an API key under *Settings → API Keys*. New accounts typically receive a small free credit balance, which is enough for this lab many times over (the lab uses well under a dollar of API credit even if you experiment heavily). When you have the key, store it in Colab using the key icon in the left sidebar — name the secret `ANTHROPIC_API_KEY` and paste the key as the value. The code cell below reads from that secret. Do not paste the key directly into the notebook; treat API keys like passwords.

**3. A few Python libraries.**
The first code cell installs them. This takes 30 to 60 seconds the first time.

In [ ]:
# Install the four libraries this lab uses.
# Run this cell once per Colab session.

!pip install --quiet pypdf sentence-transformers chromadb anthropic

print("Libraries installed. You can ignore any 'dependency conflicts' warnings — they don't affect this lab.")

In [ ]:
# Read the Anthropic API key from Colab's secret manager.
# If you are running this notebook locally instead of in Colab,
# replace this block with: os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded from Colab secrets.")
except ImportError:
    # Not running in Colab — assume the key is already in the environment.
    if 'ANTHROPIC_API_KEY' not in os.environ:
        raise RuntimeError("Set ANTHROPIC_API_KEY in your environment before continuing.")
    print("API key found in environment.")

---

# Part 1 — Loading the Dissertation

## Why this is the first step, and why it is harder than it looks

Every RAG system begins with the same problem: the source document is in a format designed for human readers (a PDF), and you need it in a format an LLM can work with (clean text, organized into addressable units). PDFs are notoriously messy to parse — they often contain headers and footers that repeat on every page, footnote numbers that interrupt sentences, page-break artifacts, and tables that text extraction handles badly.

**Analogy:** think of this stage as the *intake desk* of a library. The book has arrived. Before anyone can find anything in it, someone has to open it, flatten the pages, brush off the dust, and put it on the shelf where the catalog system can reach it. A messy intake produces a messy catalog, which produces useless searches. RAG is no different — the quality of everything downstream depends on how clean the text is here.

For this lab we will use `pypdf`, the simplest serviceable PDF parser. It is not the most sophisticated tool — for production work on heavily formatted documents you would want to look at `pdfplumber`, `unstructured`, or commercial tools like Azure Document Intelligence. But for a text-dominant dissertation, `pypdf` is more than enough, and its simplicity helps you see what is going on.

In [ ]:
# Load the dissertation PDF and extract its text, page by page.

from pypdf import PdfReader

PDF_PATH = "/content/dissertation.pdf"   # adjust if your file is named differently

reader = PdfReader(PDF_PATH)
page_count = len(reader.pages)

# Extract text from every page. We keep page numbers because we may want
# to cite them later when we get to the citation-tracking section.
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({"page": page_number, "text": text})

print(f"Loaded {page_count} pages.")
print(f"First 400 characters of page 1:\n")
print(pages[0]['text'][:400])

In [ ]:
# Quick sanity checks on what we extracted.
# A dissertation should be on the order of 150 pages and roughly 30,000 words.

total_chars = sum(len(p['text']) for p in pages)
total_words = sum(len(p['text'].split()) for p in pages)
empty_pages = sum(1 for p in pages if len(p['text'].strip()) < 50)

print(f"Total characters extracted: {total_chars:,}")
print(f"Approximate word count:     {total_words:,}")
print(f"Pages that came out nearly empty (likely image-only or formatting artifacts): {empty_pages}")

### Reflection 1 — Pause here before continuing

Look at the output of the two cells above. Three things to notice and write down (mentally or in the cell below):

1. **Does the page-1 text look clean, or are there artifacts** — header text repeated, page numbers embedded in the prose, hyphens in the middle of words from line breaks? Whatever you see is what every downstream stage will inherit. RAG systems rarely fail at the LLM step; they fail at the parsing step and the failure only becomes visible much later.

2. **How many pages came out nearly empty?** Image-heavy pages, charts, and complex tables are common culprits. A handful is normal. Many is a signal that you need a better parser.

3. **Connect this back to the dissertation's findings.** Your dissertation argues that Enterprise Architects and Technology Architects need to be more deliberately aware of data quality artifacts and measurement within their organizations. PDF parsing is a small live example of the same principle: `pypdf` will happily return text that *looks* successfully extracted, but the only way to know whether the data is actually fit for purpose is to inspect it intentionally. Quiet success is the most dangerous failure mode for downstream systems — exactly the awareness gap the dissertation set out to characterize.

---

# Part 2 — Chunking the Text

## Why we cannot just embed the whole dissertation

An LLM has a finite context window — it can only consider so many tokens at once. Even with today's large windows, you do not want to send the entire dissertation with every query. Doing so would be slow, expensive, and counterproductive: most of the dissertation is irrelevant to any given question, and stuffing irrelevant text into the prompt actively degrades the LLM's answer quality (a phenomenon researchers call *context dilution*).

So we break the document into smaller passages — *chunks* — and at query time we retrieve only the chunks most relevant to the question. Chunking is the single most consequential design decision in a RAG system. Get it right and retrieval works; get it wrong and nothing downstream can recover.

**Analogy:** think of chunking as tearing a textbook into useful pages for an open-book exam. Tear it into individual sentences and you lose context — a sentence often makes no sense without the paragraph around it. Tear it into whole chapters and the "page" you grab in the exam contains too much irrelevant material to read in time. The right chunk is roughly the unit at which a question is naturally answered: a paragraph or two, maybe a section.

For dissertation prose, **a chunk size of around 500 to 800 words with about 100 words of overlap between adjacent chunks** is a sensible starting point. The overlap matters: it means a concept that straddles a chunk boundary still appears in at least one chunk in its full context. We will use 600-word chunks with 100-word overlap.

In [ ]:
# A simple word-based chunker with overlap.
# More sophisticated chunkers split on sentences, headings, or semantic boundaries —
# but word-based with overlap is robust and a fine starting point for a learning exercise.

def chunk_text(text, chunk_size=600, overlap=100):
    '''Split text into overlapping chunks of approximately `chunk_size` words.'''
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap   # step forward, but back up by `overlap`
    return chunks

# Combine all page text into one stream, then chunk it.
# We keep a record of which page(s) each chunk roughly came from
# by tracking the running character offset.

full_text = "\n\n".join(p['text'] for p in pages)
chunks = chunk_text(full_text, chunk_size=600, overlap=100)

print(f"Produced {len(chunks)} chunks from {len(full_text.split()):,} words.")
print(f"Average chunk length (words): {sum(len(c.split()) for c in chunks) // len(chunks)}")

In [ ]:
# Inspect a chunk from the middle of the dissertation.
# Look at where it starts and ends — does it begin mid-sentence?
# Does it cut off mid-paragraph? Both are normal with this chunker.

middle = len(chunks) // 2
print(f"--- Chunk #{middle} ---\n")
print(chunks[middle][:1200])
print(f"\n[...chunk continues for another {len(chunks[middle]) - 1200} characters...]")

### Reflection 2 — On chunking tradeoffs

The chunker above is intentionally simple. Look at the middle chunk you just printed and ask yourself:

1. **Does the chunk cut off mid-thought?** If so, can a reader (or LLM) still make sense of it? This is where the 100-word overlap pays off — the next chunk will contain the resumption of the thought.

2. **Would a different chunking strategy serve the dissertation better?** Academic prose has natural boundaries — section headings, paragraph breaks, citations — that a smarter chunker could respect. The cost is complexity. The honest tradeoff in production RAG is roughly: simple chunking gets you to 80% of optimal performance with one afternoon of work; sophisticated chunking takes weeks and gets you to 90%. For many use cases, the simple version is the right call.

3. **What chunk size would you choose for, say, use cases for your internal LLM where the source documents are specific to your organization?** If you work in a highly-regulated environment, regulations are dense, internally cross-referenced, and unforgiving of context loss. You would probably want smaller chunks with more overlap, and you would want to preserve section numbers as metadata. Take a moment to think through what your answer would be.

---

# Part 3 — Turning Chunks into Embeddings

## What embeddings actually are

An embedding is a numerical representation of a piece of text — a list of (typically) several hundred floating-point numbers — produced by a neural network trained on the property that **semantically similar texts produce numerically similar vectors**.

**Analogy:** think of an embedding as a GPS coordinate, but with several hundred dimensions instead of two. A document about "water quality assessment in Washington State" sits at a particular point in this high-dimensional space. A document about "lake monitoring in the Pacific Northwest" sits near it — close in space because close in meaning. A document about "monetary policy in the Eurozone" sits very far away. The retrieval step in RAG works by converting your question into the same kind of coordinate, then finding the nearest neighbors among your stored chunks.

We will use the `all-MiniLM-L6-v2` model from the SentenceTransformers library. It is small, fast, runs on a CPU comfortably, and produces 384-dimensional embeddings. It is the sensible default for a first RAG project. Larger and more recent models (such as `bge-large-en` or proprietary embedding APIs from OpenAI, Cohere, or Voyage) produce somewhat better retrieval quality but require more compute or external API calls. The conceptual mechanics are identical.

In [ ]:
# Load the embedding model. First-time use downloads the model (~90MB).
# Subsequent uses load from cache.

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all chunks. For a dissertation-size document this takes 15-45 seconds on CPU.
print("Embedding chunks. This will take a moment...")
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print(f"\nProduced {len(chunk_embeddings)} embeddings, each of dimension {chunk_embeddings.shape[1]}.")
print(f"First chunk's embedding (first 8 numbers of 384):")
print(chunk_embeddings[0][:8])

In [ ]:
# Quick concept demonstration: similar sentences get similar embeddings.
# Run this and look at the cosine similarity scores.

import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sentences = [
    "Data quality is essential for trustworthy AI outputs.",
    "Information quality directly affects the reliability of machine learning.",   # similar meaning
    "The capital of France is Paris.",                                              # unrelated
]

sentence_embeddings = embedding_model.encode(sentences)

print("Similarities (1.0 = identical, 0.0 = unrelated):\n")
print(f"  Sentence 1 vs Sentence 2 (both about IQ/DQ for AI): {cosine_similarity(sentence_embeddings[0], sentence_embeddings[1]):.3f}")
print(f"  Sentence 1 vs Sentence 3 (DQ vs capital city):      {cosine_similarity(sentence_embeddings[0], sentence_embeddings[2]):.3f}")

### Reflection 3 — On the strange power of vector geometry

The numbers in the cell above demonstrate something genuinely strange and powerful: a 384-dimensional vector of seemingly arbitrary floating-point numbers carries enough information that two sentences with related meanings sit measurably closer together than two sentences with unrelated meanings. No one programmed the model to know that "data quality" and "information quality" are related concepts. The model learned that relationship — along with millions of others — by being trained to predict missing words in sentences across a huge corpus of text.

A few things worth sitting with:

1. **The embedding model has never read the dissertation.** It cannot tell you what the dissertation says. But it has learned enough about the structure of language and meaning that it can reliably tell whether two passages from the dissertation are about similar topics. That partial understanding is exactly what we need for retrieval.

2. **This is also where bias enters.** The model learned from a particular slice of internet text. Concepts well-represented in that training data (English-language technical writing, for example) embed cleanly; concepts under-represented (low-resource languages, domain-specific jargon the model didn't see) embed less reliably. For your specific organization's vocabulary, you may find this matters in practice.

3. **The landing.** This is the kind of concept where a clear analogy lands better than mathematical detail. The GPS-with-many-dimensions framing is one good way to explain it. Another is "smart synonyms" — embeddings let you find passages that are *about the same thing* even when they don't share any of the same words. Both framings are honest.

---

# Part 4 — Storing and Querying with ChromaDB

## What a vector store does (and why you need one)

In principle, you could store the embeddings as a plain Python list and search them with a `for` loop. For a single dissertation with a few hundred chunks, that would work. For anything bigger — thousands of documents, millions of chunks — naive search becomes too slow. A vector store is a specialized database that uses clever data structures (typically approximate-nearest-neighbor indices like HNSW) to find the closest matches across millions of vectors in milliseconds.

**Analogy:** think of a vector store as a *smart filing cabinet* where the drawers are organized by meaning rather than by alphabetical order. Walk up with a question, the cabinet opens the drawer of "things related to your question," and you read what's inside.

We will use **ChromaDB**, an open-source vector store that runs entirely in-process. It is the simplest choice for learning. For production work at scale you would look at Pinecone, Weaviate, Qdrant, or Milvus — but the API patterns are nearly identical.

In [ ]:
# Set up Chroma with our dissertation chunks.

import chromadb

# Create an in-memory Chroma client. (Use chromadb.PersistentClient(path=...) to persist between sessions.)
client = chromadb.Client()
collection = client.create_collection(name="dissertation")

# Add chunks. Chroma will use its default embedder if we don't provide embeddings,
# but we already computed them with sentence-transformers, so we hand them in.
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
)

print(f"Stored {collection.count()} chunks in the vector store.")

In [ ]:
# Query the vector store. Ask a question, get the top-k most relevant chunks.

def retrieve(question, k=3):
    '''Return the top-k chunks most relevant to the question.'''
    question_embedding = embedding_model.encode([question])[0].tolist()
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=k,
    )
    return results['documents'][0]   # the actual chunk text

# Try it with a question your dissertation should be able to answer well.
question = "What is the level of understanding of data quality among Enterprise Architects, according to this research?"

retrieved_chunks = retrieve(question, k=3)

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"--- Retrieved chunk {i} ---")
    print(chunk[:400])
    print()

### Reflection 4 — On what retrieval is doing, and what it isn't

Look carefully at what was just returned. The vector store gave you the three chunks the dissertation whose embeddings are mathematically closest to your question's embedding. A few things to notice:

1. **The retrieval has no idea whether the chunks actually *answer* the question.** It only knows they discuss similar concepts. This is an important honesty: retrieval is the *raw material* the LLM will reason over, not the answer itself. A good retrieval can still produce a bad answer if the LLM mishandles it. A poor retrieval almost always produces a poor answer regardless of how good the LLM is.

2. **The order matters.** The first chunk is the closest match. If your top result is irrelevant, your downstream answer will be too. In production systems, much engineering effort goes into *re-ranking* — taking the top 20 retrieval results and re-scoring them with a more careful (but slower) model to pick the truly best three.

3. **Try a question whose answer is *not* in the dissertation.** A question about, say, the molecular structure of caffeine. The vector store will still return three chunks — the ones with the highest similarity scores — even though none of them are relevant. This is one of the failure modes RAG systems must explicitly handle, and we will see it in Part 6.

---

# Part 5 — Wiring in the LLM

## The whole RAG pattern, finally assembled

We now have the three ingredients: a question, a way to find relevant passages from the dissertation, and an LLM that can reason over text. The RAG pattern composes them in a specific order:

1. Take the user's question.
2. Retrieve the top-k most relevant chunks from the vector store.
3. Construct a prompt that instructs the LLM: *here are some passages from a source document, here is a question, answer the question using only the passages.*
4. Send the prompt to the LLM.
5. Return the response.

The *prompt construction* is where the engineering judgment lives. We want to be explicit that the LLM should answer only from the provided passages, should acknowledge when the passages don't contain the answer, and should write at an appropriate level for the audience. We will use Claude (Anthropic's model) here, though the pattern is identical for any LLM API.

In [ ]:
# Set up the Anthropic client and define the RAG function.

import anthropic

claude = anthropic.Anthropic()   # picks up ANTHROPIC_API_KEY from the environment

RAG_SYSTEM_PROMPT = '''You are a careful research assistant answering questions about a doctoral dissertation.

You will be given passages from the dissertation followed by a question. Follow these rules strictly:

1. Answer the question using ONLY information present in the provided passages.
2. If the passages do not contain enough information to answer the question, say so plainly. Do not invent or speculate.
3. Quote directly from the passages sparingly and only when the exact wording matters. Otherwise, paraphrase in your own words.
4. Write in a clear, scholarly tone appropriate for an academic audience.
'''

def rag_answer(question, k=4):
    '''The full RAG pipeline: retrieve relevant chunks, ask the LLM.'''
    retrieved = retrieve(question, k=k)

    passages_block = "\n\n".join([f"[Passage {i+1}]\n{chunk}" for i, chunk in enumerate(retrieved)])

    user_prompt = f'''Passages from the dissertation:

{passages_block}

Question: {question}'''

    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system=RAG_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    return response.content[0].text

# Try it out.
question = "What gaps does this dissertation identify in how Enterprise Architecture practices integrate data quality frameworks and specifications?"
answer = rag_answer(question)
print(answer)

### Reflection 5 — On what just happened

You just built a working RAG system. Read the answer the LLM produced and ask yourself:

1. **Does the answer accurately reflect what your dissertation actually says?** If you read through the source document, you are positioned to fact-check the system. This is one of the genuinely fun parts of building RAG over literary works of your own choosing — you can immediately tell whether the system is grounded or whether it has drifted.

2. **Did the LLM stay within the passages, or did it pull in general knowledge?** Look for signs of either. A well-behaved RAG answer will not introduce concepts that aren't in the retrieved passages. If it does, your system prompt may need to be more emphatic, or your retrieval may have missed the relevant chunks.

3. **Try a question you know the dissertation does NOT address** — the molecular structure of caffeine, the rules of basketball, the price of tea in 1840. A well-instructed RAG system should say "the passages don't address this." If your system makes something up, you have just observed an *unfaithful answer* — a serious RAG failure mode that real production systems work hard to detect and prevent.

---

# Part 6 — With RAG vs Without RAG

## Why the comparison matters

The most compelling demonstration of what RAG actually adds is to ask the same question two ways: once with retrieved passages from the dissertation, and once with the bare question and no retrieval at all. The LLM has no specific knowledge of the dissertation — it was trained on the public internet, where the dissertation may or may not appear. Watching the answers diverge is the cleanest possible way to internalize what RAG does.

This is also the comparison that lets you explain RAG to a non-technical audience. *"Here is what the AI says when it's working from its training data. Here is what it says when we ground it in our actual documents. Notice the difference."* That two-shot demonstration is more persuasive than any architectural diagram.

In [ ]:
# Compare grounded (RAG) answer with ungrounded (raw LLM) answer.

def raw_answer(question):
    '''Ask the LLM with no retrieval — just the bare question.'''
    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

# Pick a question that is specific to your dissertation —
# something the LLM would NOT know about from its general training.
question = "What did the survey in this dissertation find regarding respondents' awareness of whether their own organizations are measuring data quality, and what does the research recommend about training and education for Enterprise Architects?"

print("=" * 70)
print("WITHOUT RAG (raw LLM, working from training data only):")
print("=" * 70)
print(raw_answer(question))

print("\n" + "=" * 70)
print("WITH RAG (grounded in dissertation passages):")
print("=" * 70)
print(rag_answer(question))

### Reflection 6 — On grounding, and why your dissertation argues for it

Look carefully at both answers. The differences should be striking. The ungrounded answer will be plausible-sounding general prose about data quality and Enterprise Architecture, but it cannot reference the specific survey respondents, the specific findings about EA practitioner awareness, or the specific recommendations about training and education. The grounded answer can.

This is exactly the kind of failure the dissertation argues organizations are unprepared for. *Plausible* is not the same as *accurate*. *Well-formed prose* is not the same as *grounded in a source of truth*. The ungrounded answer reads as confident and fluent, yet has no anchor to any verifiable source. The dissertation's recommendation — that organizations need to deliberately measure data quality and integrate it into how they architect their systems — is the upstream organizational practice that, applied technically downstream, looks like RAG.

Two implications worth carrying:

1. **You can demonstrate this difference live, with this notebook, on questions about a literary work that you've read and understand.** It is one of the most compelling demos a person can show because it requires no setup beyond opening the notebook. If an opportunity arises along the lines of *"show me something you've built,"* this is what you show.

2. **The dissertation-and-implementation arc is theory turned into practice.** Most people can speak to *either* the theory of trustworthy AI *or* the practice of building RAG. You can speak to both, and you can demonstrate the connection by showing how a technique solves the exact problem the scholarly work identified. That coherence is unusual and worth surfacing explicitly if the opportunity ever presents itself.

---

# Part 7 (Bonus) — Citation Tracking, Because Provenance Matters

## Why this small addition is actually a big deal

A core principle in information quality — and in the dissertation specifically — is that data without a known source is data without trust. A RAG system that returns answers without telling you which passages they came from is the LLM equivalent of an unsourced claim. For internal staff use that may be tolerable; for regulatory, scientific, or public-facing use it is not.

The fix is small in code and large in meaning: modify the RAG function so it returns not only the answer but the passages used to generate it. The downstream user can verify the claim against the source. This is *data provenance* applied to LLM outputs.

**Analogy:** the difference between *"the report says revenue grew"* and *"the report says revenue grew, see page 14, paragraph 3"* is the difference between an unverifiable assertion and a checkable claim. RAG with citations is the second; RAG without is the first.

In [ ]:
# A RAG function that returns sources alongside the answer.

def rag_answer_with_sources(question, k=4):
    '''Return both the LLM answer and the source passages that informed it.'''
    retrieved = retrieve(question, k=k)

    passages_block = "\n\n".join([f"[Passage {i+1}]\n{chunk}" for i, chunk in enumerate(retrieved)])

    user_prompt = f'''Passages from the dissertation:

{passages_block}

Question: {question}

When you answer, cite the passage numbers you used in square brackets, like [Passage 1] or [Passage 2, 3].'''

    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system=RAG_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    return {
        "answer": response.content[0].text,
        "sources": retrieved,
    }

# Try it.
question = "What gap between data-quality theory and practice does this dissertation argue exists?"

result = rag_answer_with_sources(question)

print("ANSWER:")
print(result['answer'])
print("\n" + "=" * 70)
print("SOURCES (the actual passages the LLM had to work from):")
print("=" * 70)
for i, source in enumerate(result['sources'], start=1):
    print(f"\n--- Passage {i} ---")
    print(source[:500] + ("..." if len(source) > 500 else ""))

### Reflection 7 — On provenance as a feature, not a footnote

You now have a RAG system that returns auditable answers. Notice that the citation behavior is achieved through *prompt engineering* (asking the LLM to cite passages) rather than through architecture — Claude is reliable enough at instruction-following that the cite-the-passages request usually works without additional machinery. For production systems where reliability matters more, you would add programmatic verification: parse the response for citation markers, confirm each citation refers to an actual retrieved passage, flag responses with missing citations.

When discussing RAG, this is the kind of small extension that demonstrates judgment. *"I extended the basic RAG pattern to include citation tracking, because in Information Quality, an unsourced answer is operationally indistinguishable from a fabricated one."* That sentence shows you understand both the technical pattern and the organizational principle behind it.

---

# Closing — What You've Built and How to Talk About It

## What is now true that wasn't true before

You have built a working Retrieval-Augmented Generation system, end-to-end, that:

- Ingests a real PDF document
- Chunks it intelligently with overlap
- Embeds the chunks using a peer-reviewed open-source model
- Stores them in a vector database
- Retrieves the most relevant chunks for any given question
- Constructs a grounded prompt and queries an LLM
- Returns answers with verifiable source citations

You did this in roughly 90 minutes. The total code, stripped of explanation, is under 100 lines. **This is what AI engineering looks like at the entry-to-mid level** — composing well-engineered libraries into systems that solve real problems, not inventing new transformer architectures from scratch.

## How to talk about this with your friends and peers

If questions manifest such as — *"tell me about a RAG system you've built,"* or *"talk me through how you would approach a document-question-answering project,"* or *"do you have anything you can show me?"* — this notebook is your answer. A few framings worth preparing:

**On what you built.** *"I built a RAG system over a dissertation as a learning exercise. The interesting design decisions were chunk sizing — I went with 600 words and 100-word overlap, which is appropriate for academic prose but I'd reduce both for something like environmental regulations — and citation tracking, which I added because the study and practice of Information and Data Quality makes me allergic to unsourced answers."*

**On what you learned.** *"The thing the lab made concrete for me is how much of RAG data quality lives in the parsing and chunking stages, not the LLM stage. The LLM is the part everyone is excited about, but it's downstream of decisions that determine whether the LLM has anything useful to work with."*

**On what you would extend.** *"The next things on my list are re-ranking with a cross-encoder for retrieval quality, programmatic verification of citations to catch hallucinations, and an evaluation harness so I can measure changes rather than eyeball them. The notebook is a starting point, not a finished system."*

**On the connection to the dissertation.** *"There's an arc here I find satisfying. The dissertation argued that Enterprise Architects often lack awareness of data quality measurement and that gaps exist in how data quality frameworks are integrated into EA practice — with downstream implications for any AI system the enterprise deploys. RAG is one of the principal technical patterns for grounding LLMs in authoritative source documents, which is one downstream answer to the upstream organizational gap the dissertation describes. Building RAG over the dissertation itself was, in a small way, applying my own recommendation on my peer-reviewed text."*

## Good luck and may the force be with you!
